In [1]:
import pandas as pd
import numpy as np
import glob
from scipy.stats import ttest_ind, ttest_1samp, fisher_exact
import scipy
from tqdm import tqdm

PROJDATA = '/scratch/hi387/SemanticScholar'
CLEANDATA = '../data'
SCISCI = '/scratch/fl1092/SciSciNet/v2'

DATA = '/scratch/fl1092/Email_project/semantic_scholar_data'

# Load data

## Load SciSciNet

In [2]:
%%time
paperDOI2024 = pd.read_csv(f'{SCISCI}/cleaned/PaperDOI.csv') # Paper DOIs from SciSciNet
paperDOI2025 = pd.read_csv(f'{SCISCI}/extension_2025/PaperDOI_ext_2025.csv', usecols=['PaperID','doi']) # Paper DOIs from OpenAlex (to extend the SciSciNet dataset to 2025)

CPU times: user 2min 59s, sys: 16.6 s, total: 3min 15s
Wall time: 3min 24s


In [3]:
%%time
paperDOI = pd.concat([paperDOI2024, paperDOI2025], ignore_index=True, sort=False)
del paperDOI2024
del paperDOI2025

CPU times: user 4.75 s, sys: 1.04 s, total: 5.78 s
Wall time: 6.01 s


## Load Semantic Scholar

In [4]:
%%time
citations = pd.read_csv(f'{DATA}/CitationContext.csv', usecols=['citingcorpusid','citedcorpusid']).drop_duplicates() # all citation contexts

CPU times: user 10.4 s, sys: 371 ms, total: 10.7 s
Wall time: 11.8 s


In [5]:
%%time
allCitingCited = (
    pd.concat(
        [
            citations[['citingcorpusid']].rename(columns={'citingcorpusid':'CorpusId'}),
            citations[['citedcorpusid']].rename(columns={'citedcorpusid':'CorpusId'})
        ], ignore_index=True, sort=False
    )
    .drop_duplicates()
)

CPU times: user 289 ms, sys: 978 μs, total: 290 ms
Wall time: 320 ms


In [6]:
%%time
semanticsPubMed = (
    pd.read_csv(f'{PROJDATA}/clean_data/PaperExternalIDs.csv', usecols=['CorpusId','PubMed','PubMedCentral'])
    .assign(BothNA=lambda df: df.PubMed.isna() & df.PubMedCentral.isna())
    .query('BothNA==False')
)

CPU times: user 1min 16s, sys: 7.75 s, total: 1min 24s
Wall time: 1min 32s


In [7]:
%%time
semanticsDOI = (
    pd.read_csv(f'{PROJDATA}/clean_data/PaperExternalIDs.csv', usecols=['CorpusId','DOI'])
    .dropna()
)

CPU times: user 2min 21s, sys: 12.8 s, total: 2min 34s
Wall time: 2min 41s


In [8]:
%%time
semanticsMAG = (
    pd.read_csv(f'{PROJDATA}/clean_data/PaperExternalIDs.csv', usecols=['CorpusId','MAG'])
    .dropna()

    .assign(MAG=lambda df: df.MAG.astype(int))
)

CPU times: user 1min 21s, sys: 6.82 s, total: 1min 27s
Wall time: 1min 31s


In [ ]:
semanticsMAG.shape # 181357793

In [9]:
semanticsMAG.head()

,MAG,CorpusId
0,2342209090,75631345
1,2593466797,157903623
2,1506287221,106813104
3,2933815690,132811066
7,2009373353,109657477


# Match

## MAG, using ID

In [10]:
%%time
magMatched = (
    allCitingCited.merge(semanticsMAG, on='CorpusId')
    .assign(PaperID=lambda df: df.MAG.astype(int))
    .drop('MAG', axis=1)
)

CPU times: user 58.1 s, sys: 6.68 s, total: 1min 4s
Wall time: 1min 7s


In [11]:
magMatched.head()

,CorpusId,PaperID
0,219401867,3033190048
1,51718194,2789451386
2,219589053,3033485143
3,209488282,2413941879
4,238769689,3193230284


In [12]:
unmatched = allCitingCited[~allCitingCited.CorpusId.isin(magMatched.CorpusId)]

## OpenAlex, using DOI

In [13]:
semanticsDOI.head()

,CorpusId,DOI
0,75631345,10.1016/S0016-5085(16)33009-8
4,281728455,10.33764/2618-981x-2025-2-1-90-95
5,247633687,10.1007/s11069-022-05314-x
7,109657477,10.1016/0021-9290(85)90741-9
9,158860007,10.1002/APP5.206


In [14]:
paperDOI.head()

,doi,PaperID
0,https://doi.org/10.1007/978-1-4471-0761-3_15,112393110
1,https://doi.org/10.48550/arxiv.2208.11981,4293326940
2,https://doi.org/10.5281/zenodo.10497732,4391245458
3,https://doi.org/10.1075/tsl.32.23fle,2491627364
4,https://doi.org/10.1515/bgsl.1973.1973.95.333,1978381389


In [15]:
%%time
doiMatched = (
    unmatched.merge(semanticsDOI, on='CorpusId')
    .assign(doi=lambda df: df.DOI.apply(lambda x: f'https://doi.org/{x.lower()}'))
    .merge(paperDOI, on='doi')
)

CPU times: user 5min 43s, sys: 12.3 s, total: 5min 55s
Wall time: 6min 1s


In [16]:
unmatched = (
    unmatched[~unmatched.CorpusId.isin(doiMatched.CorpusId)]
)

## OpenAlex, using API

In [ ]:
import json
import os
import time
from typing import Any, Dict, Optional, Tuple

import pandas as pd
import requests


OPENALEX_BASE = "https://api.openalex.org/works"
USER_AGENT = "paper-id-lookup/1.0 (mailto:michael.liu916@gmail.com)"


def normalize_doi(doi: Optional[Any]) -> Optional[str]:
    if pd.isna(doi):
        return None
    doi = str(doi).strip()
    if not doi:
        return None

    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("doi.org/", "")
    doi = doi.replace("doi:", "")
    doi = doi.strip()
    return doi or None


def normalize_pmid(pmid: Optional[Any]) -> Optional[str]:
    if pd.isna(pmid):
        return None
    pmid = str(pmid).strip()
    if not pmid:
        return None

    pmid = pmid.replace("pmid:", "").strip()
    return pmid or None


def normalize_pmcid(pmcid: Optional[Any]) -> Optional[str]:
    if pd.isna(pmcid):
        return None
    pmcid = str(pmcid).strip()
    if not pmcid:
        return None

    pmcid = pmcid.upper().replace("PMCID:", "").strip()
    if not pmcid.startswith("PMC"):
        pmcid = f"PMC{pmcid}"
    return pmcid or None


def fetch_openalex_work(
    external_id: str,
    session: requests.Session,
    api_key: Optional[str] = None,
    timeout: int = 30,
) -> Optional[Dict[str, Any]]:
    url = f"{OPENALEX_BASE}/{external_id}"
    params = {}
    if api_key:
        params["api_key"] = api_key

    response = session.get(url, params=params, timeout=timeout)

    if response.status_code == 404:
        return None

    response.raise_for_status()
    return response.json()


def lookup_row_in_openalex(
    row: pd.Series,
    session: requests.Session,
    api_key: Optional[str] = None,
    sleep_seconds: float = 0.1,
) -> Dict[str, Any]:
    doi = normalize_doi(row.get("DOI"))
    pmid = normalize_pmid(row.get("PubMed"))
    pmcid = normalize_pmcid(row.get("PubMedCentral"))

    result = {
        "CorpusId": row.get("CorpusId"),
        "DOI": row.get("DOI"),
        "PubMed": row.get("PubMed"),
        "PubMedCentral": row.get("PubMedCentral"),
        "openalex_id": None,
        "openalex_short_id": None,
        "matched_by": None,
        "openalex_display_name": None,
        "lookup_status": "not_found",
        "error": None,
    }

    candidates = []
    if doi:
        candidates.append(("DOI", f"https://doi.org/{doi}"))
    if pmid:
        candidates.append(("PubMed", f"pmid:{pmid}"))
    if pmcid:
        candidates.append(("PubMedCentral", pmcid))

    for matched_by, external_id in candidates:
        try:
            work = fetch_openalex_work(external_id, session=session, api_key=api_key)
        except requests.HTTPError as e:
            result["lookup_status"] = "http_error"
            result["error"] = f"HTTP {e.response.status_code}"
            return result
        except requests.RequestException as e:
            result["lookup_status"] = "request_error"
            result["error"] = f"{type(e).__name__}: {e}"
            return result

        if work:
            openalex_id = work.get("id")
            result["openalex_id"] = openalex_id
            result["openalex_short_id"] = (
                openalex_id.rsplit("/", 1)[-1] if openalex_id else None
            )
            result["matched_by"] = matched_by
            result["openalex_display_name"] = work.get("display_name")
            result["lookup_status"] = "found"
            return result

        time.sleep(sleep_seconds)

    return result


def append_jsonl(path: str, record: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def find_openalex_ids_and_stream_to_jsonl(
    records: pd.DataFrame,
    output_jsonl_path: str,
    api_key: Optional[str] = None,
    sleep_seconds: float = 0.1,
    resume: bool = False,
) -> pd.DataFrame:
    """
    Process a pandas DataFrame with columns:
      - CorpusId
      - DOI
      - PubMed
      - PubMedCentral

    Writes one JSON line to disk immediately after each row is processed.

    Parameters
    ----------
    records : pd.DataFrame
        Input dataframe.
    output_jsonl_path : str
        Path to output JSONL file.
    api_key : str | None
        Optional OpenAlex API key.
    sleep_seconds : float
        Delay between fallback attempts.
    resume : bool
        If True, skip CorpusId values already present in the JSONL file.

    Returns
    -------
    pd.DataFrame
        DataFrame of processed results.
    """
    required_cols = ["CorpusId", "DOI", "PubMed", "PubMedCentral"]
    missing = [c for c in required_cols if c not in records.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    already_done = set()
    if resume and os.path.exists(output_jsonl_path):
        with open(output_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    already_done.add(obj.get("CorpusId"))
                except json.JSONDecodeError:
                    continue

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    results = []

    try:
        for _, row in records.iterrows():
            corpus_id = row.get("CorpusId")

            if resume and corpus_id in already_done:
                continue

            result = lookup_row_in_openalex(
                row=row,
                session=session,
                api_key=api_key,
                sleep_seconds=sleep_seconds,
            )

            append_jsonl(output_jsonl_path, result)
            results.append(result)

            if result['lookup_status'] == "http_error":
                sleep_seconds *= 2
                print("Increase sleep to", sleep_seconds, result)
                
    finally:
        session.close()

    return pd.DataFrame(results)

In [ ]:
%%time
unmatchedDOIPubMed = (
    unmatched
    
    .merge(semanticsDOI, on='CorpusId', how='left')
    .merge(semanticsPubMed, on='CorpusId', how='left')
)

In [ ]:
# Example usage:
results_df = find_openalex_ids_and_stream_to_jsonl(
    records=unmatchedDOIPubMed,
    output_jsonl_path=f"{DATA}/openalex_matched_results.jsonl",
    api_key=None,
    sleep_seconds=.1,
    resume=True,
)

In [17]:
APImatched = (
    pd.read_json(f'{DATA}/openalex_matched_results.jsonl', lines=True)
    .dropna(subset=['openalex_id'])
    .assign(PaperID=lambda df: df.openalex_short_id.apply(lambda x: int(x.replace('W',''))))
    [['CorpusId','PaperID']]
)

In [18]:
APImatched.head(2)

,CorpusId,PaperID
5,277220999,4408725347
6,278747638,4410495445


## Gather matched

In [19]:
allMatched = (
    pd.concat([
        magMatched, doiMatched.drop(['DOI', 'doi'], axis=1), APImatched
    ], ignore_index=True, sort=False)
)

In [23]:
allMatched.CorpusId.nunique()

1390123

In [24]:
allMatched.head()

,CorpusId,PaperID
0,219401867,3033190048
1,51718194,2789451386
2,219589053,3033485143
3,209488282,2413941879
4,238769689,3193230284


In [ ]:
allMatched.to_csv(f'{DATA}/PapersMatchedUsingOpenAlex.csv', index=False)

In [25]:
1390123/1606522

0.8652996971096567

# Paper region

In [ ]:
%%time
paperGlobalSouth = pd.read_csv(f'{SCISCI}/derived/PaperGlobalSouthAuthorCount.csv')
paperGlobalNorth = pd.read_csv(f'{SCISCI}/derived/PaperGlobalNorthAuthorCount.csv')
paperCountry = pd.read_csv(f'{SCISCI}/derived/PaperCountryAuthorCount.csv')
paperSubRegion = pd.read_csv(f'{SCISCI}/derived/PaperSubRegionAuthorCount.csv')

In [ ]:
continens = pd.read_csv('/scratch/fl1092/data_common/continents2.csv')

In [ ]:
allMatchedRegion = (
    allMatched
    
    .merge(paperGlobalSouth.rename(columns={'Percent':'SouthPercent'}), on='PaperID')
    .assign(AllGlobalSouth=lambda df: df.SouthPercent == 1) # all from Global South
    .drop(['Total'], axis=1)
    
    .merge(paperGlobalNorth.rename(columns={'Percent':'NorthPercent'}), on='PaperID')
    .assign(NonGlobalNorth=lambda df: df.NorthPercent == 0) # none from Global North
    .drop(['Total'], axis=1)
    
    .assign(NoGNAllGS=lambda df: (df.SouthPercent == 1) & (df.NorthPercent == 0)) # none from Global North
)

In [ ]:
allMatchedRegion.to_csv(f'{DATA}/MatchedPaperRegion.csv', index=False)

In [ ]:
allMatchedRegion = (
    allMatched
    
    .merge(paperSubRegion, on='PaperID') # subregions
    .query('Percent >= 0.5')

    .sort_values(by=['Percent'], ascending=False)
    .drop_duplicates(subset=['CorpusId'], keep='first')
)

In [ ]:
allMatchedRegion.to_csv(f'{DATA}/MatchedPaperSubRegion.csv', index=False)

In [ ]:
allMatchedCountry = (
    allMatched
    
    .merge(paperCountry, on='PaperID')
    .query('Percent >= 0.5')

    .sort_values(by=['Percent'], ascending=False)
    .drop_duplicates(subset=['CorpusId'], keep='first')
)

In [ ]:
allMatchedCountry.to_csv(f'{DATA}/MatchedPaperCountry.csv', index=False)